# Notebook 7 — Context + Strong FiLM RoBERTa

This notebook trains the strongest context-aware identity-conditioned model.

Input:
```text
Retrieved Wikimedia context + comment
```

Identity mechanism:
```text
subgroup_id → subgroup embedding → gamma, beta
h_film = gamma(subgroup) * h_cls + beta(subgroup)
```

This combines semantic context with Strong FiLM identity conditioning.


In [9]:
import ast
import json
import random
import itertools
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, confusion_matrix, classification_report
from scipy.spatial.distance import jensenshannon
from scipy.stats import entropy, pearsonr, spearmanr

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 220)

RANDOM_SEED = 42
MODEL_NAME = "roberta-base"
NUM_LABELS = 3

MAX_LENGTH = 384
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 1

EPOCHS = 7
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
GRAD_CLIP = 1.0

SUBGROUP_DIM = 32
FILM_HIDDEN_DIM = 128
FILM_STRENGTH = 2.0
DROPOUT = 0.1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = Path("/home/shayan/Distributional-Hate-Speech-Prediction/notebooks/data")
CONTEXT_PATH = Path("/home/shayan/Distributional-Hate-Speech-Prediction/notebooks/women_models/women_context_artifacts/women_context_mapped_examples.parquet")

OUTPUT_DIR = Path("context_strong_film_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Device:", DEVICE)
print("Context file:", CONTEXT_PATH)
print("Output directory:", OUTPUT_DIR.resolve())
print("MAX_LENGTH:", MAX_LENGTH)
print("BATCH_SIZE:", BATCH_SIZE)
print("FILM_STRENGTH:", FILM_STRENGTH)


Device: cuda
Context file: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/women_models/women_context_artifacts/women_context_mapped_examples.parquet
Output directory: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/women_models/context_strong_film_outputs
MAX_LENGTH: 384
BATCH_SIZE: 8
FILM_STRENGTH: 2.0


In [10]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)


## Load and sanity-check context data

In [11]:
context_df = pd.read_parquet(CONTEXT_PATH)

print("Context dataset:", context_df.shape)
display(context_df.head(2))

required_columns = [
    "comment_id",
    "split",
    "subgroup",
    "text",
    "target_distribution",
    "target_majority_label",
    "context_input_text",
    "retrieved_article_titles",
    "retrieved_summaries",
]

missing = [col for col in required_columns if col not in context_df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Rows by split:")
display(context_df["split"].value_counts())

print("Rows by subgroup:")
display(context_df["subgroup"].value_counts())

print("Target majority distribution:")
display(context_df["target_majority_label"].value_counts(normalize=True).sort_index())

print("Unique context inputs:", context_df["context_input_text"].nunique(), "/", len(context_df))
print("Unique comments:", context_df["text"].nunique())

print("\nExample context:")
print("=" * 100)
print(context_df.iloc[0]["context_input_text"])


Context dataset: (7962, 16)


,experiment,comment_id,split,subgroup,subgroup_count,text,target_distribution,target_majority_label,target_expected_label,retrieved_article_titles,retrieved_page_urls,retrieved_similarities,retrieved_summaries,context_input_text,tweet_token_length,context_input_token_length
0,women,6,train,men,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),"[1.0, 0.0, 0.0]",0,0.0,[Male privilege],[https://en.wikipedia.org/wiki/Male_privilege],[0.10770449787378311],"[In patriarchal societies, males hold primary power and predominate in roles of leadership, moral authority, social privilege, and control of property, granting them economic, political, social, educational, and prac...",### COMMENT TO CLASSIFY\nFirst off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;)\n\n### ANNOTATOR GENDER\nmen\n\n### RETRIEVED BAC...,35,195
1,women,6,train,women,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),"[1.0, 0.0, 0.0]",0,0.0,[Misogyny],[https://en.wikipedia.org/wiki/Misogyny],[0.07197389751672745],"[Misogyny is a form of sexism that perpetuates women's subordinate status in patriarchal societies. It can manifest in obvious and subtle ways, including violence against women, sexual harassment, and exclusion from ...",### COMMENT TO CLASSIFY\nFirst off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;)\n\n### ANNOTATOR GENDER\nwomen\n\n### RETRIEVED B...,35,189


Rows by split:


split
train         5565
validation    1224
test          1173
Name: count, dtype: int64

Rows by subgroup:


subgroup
women         3919
men           3903
non_binary     140
Name: count, dtype: int64

Target majority distribution:


target_majority_label
0    0.677594
1    0.073851
2    0.248556
Name: proportion, dtype: float64

Unique context inputs: 7958 / 7962
Unique comments: 3951

Example context:
### COMMENT TO CLASSIFY
First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;)

### ANNOTATOR GENDER
men

### RETRIEVED BACKGROUND
Male privilege:
In patriarchal societies, males hold primary power and predominate in roles of leadership, moral authority, social privilege, and control of property, granting them economic, political, social, educational, and practical advantages over women. These privileges are often invisible to holders, leading them to attribute their status to individual merits rather than unearned advantages. The ideal masculine norm varies by society but is often associated with being white, heterosexual, stoic, wealthy, strong, tough, competitive, and autonomous. Men who deviate from this norm may not benefit fully from privilege, while those who conform to it are more likely to receive rewards and recognit

In [12]:
def safe_len(x):
    if isinstance(x, list):
        return len(x)
    if isinstance(x, np.ndarray):
        return len(x)
    if isinstance(x, str):
        try:
            return len(ast.literal_eval(x))
        except Exception:
            return np.nan
    return np.nan

print("Retrieved article count per row:")
display(context_df["retrieved_article_titles"].apply(safe_len).value_counts(dropna=False))

print("Retrieved summary count per row:")
display(context_df["retrieved_summaries"].apply(safe_len).value_counts(dropna=False))


Retrieved article count per row:


retrieved_article_titles
1    7962
Name: count, dtype: int64

Retrieved summary count per row:


retrieved_summaries
1    7962
Name: count, dtype: int64

## Subgroup IDs and splits

In [13]:
subgroup_to_id = {
    subgroup: idx
    for idx, subgroup in enumerate(sorted(context_df["subgroup"].unique()))
}

id_to_subgroup = {
    idx: subgroup
    for subgroup, idx in subgroup_to_id.items()
}

context_df["subgroup_id"] = context_df["subgroup"].map(subgroup_to_id)

print("Subgroup mapping:")
print(subgroup_to_id)

train_df = context_df[context_df["split"] == "train"].reset_index(drop=True)
val_df = context_df[context_df["split"].isin(["validation", "val", "dev"])].reset_index(drop=True)
test_df = context_df[context_df["split"] == "test"].reset_index(drop=True)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

assert len(train_df) > 0
assert len(val_df) > 0
assert len(test_df) > 0

display(pd.crosstab(context_df["split"], context_df["subgroup"]))


Subgroup mapping:
{'men': 0, 'non_binary': 1, 'women': 2}
Train: (5565, 17)
Val: (1224, 17)
Test: (1173, 17)


subgroup,men,non_binary,women
split,,,
test,576,16,581
train,2723,108,2734
validation,604,16,604


## Token length check

In [14]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def count_tokens(text: str) -> int:
    return len(
        tokenizer(
            text,
            truncation=False,
            add_special_tokens=True,
        )["input_ids"]
    )

if "context_input_token_length" not in context_df.columns:
    context_df["context_input_token_length"] = context_df["context_input_text"].apply(count_tokens)

length_summary = {
    "mean": context_df["context_input_token_length"].mean(),
    "median": context_df["context_input_token_length"].median(),
    "p95": context_df["context_input_token_length"].quantile(0.95),
    "max": context_df["context_input_token_length"].max(),
    "pct_over_384": float((context_df["context_input_token_length"] > 384).mean()),
    "pct_over_512": float((context_df["context_input_token_length"] > 512).mean()),
}

display(pd.DataFrame([length_summary]))
print("Rows over MAX_LENGTH:", (context_df["context_input_token_length"] > MAX_LENGTH).sum())


,mean,median,p95,max,pct_over_384,pct_over_512
0,170.833459,170.0,223.0,296,0.0,0.0


Rows over MAX_LENGTH: 0


## Dataset

In [15]:
def parse_distribution(value):
    if isinstance(value, np.ndarray):
        return value.astype(float).tolist()
    if isinstance(value, list):
        return [float(x) for x in value]
    if isinstance(value, str):
        parsed = ast.literal_eval(value)
        return [float(x) for x in parsed]
    raise TypeError(f"Unsupported distribution type: {type(value)}")


class ContextFiLMDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, tokenizer, max_length: int):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx: int):
        row = self.dataframe.iloc[idx]

        encoding = self.tokenizer(
            row["context_input_text"],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "subgroup_id": torch.tensor(row["subgroup_id"], dtype=torch.long),
            "target_distribution": torch.tensor(parse_distribution(row["target_distribution"]), dtype=torch.float),
        }


train_dataset = ContextFiLMDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = ContextFiLMDataset(val_df, tokenizer, MAX_LENGTH)
test_dataset = ContextFiLMDataset(test_df, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

batch = next(iter(train_loader))
{k: v.shape for k, v in batch.items()}


{'input_ids': torch.Size([8, 384]),
 'attention_mask': torch.Size([8, 384]),
 'subgroup_id': torch.Size([8]),
 'target_distribution': torch.Size([8, 3])}

## Model

In [16]:
class ContextStrongFiLMRoBERTa(nn.Module):
    def __init__(
        self,
        model_name: str,
        num_subgroups: int,
        subgroup_dim: int = 32,
        film_hidden_dim: int = 128,
        film_strength: float = 2.0,
        num_labels: int = 3,
        dropout: float = 0.1,
    ):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.film_strength = film_strength

        self.subgroup_embedding = nn.Embedding(num_subgroups, subgroup_dim)

        self.film_generator = nn.Sequential(
            nn.Linear(subgroup_dim, film_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(film_hidden_dim, hidden_size * 2),
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_labels),
        )

    def forward(self, input_ids, attention_mask, subgroup_id):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        subgroup_embedding = self.subgroup_embedding(subgroup_id)

        film_params = self.film_generator(subgroup_embedding)
        gamma, beta = film_params.chunk(2, dim=-1)

        gamma = 1.0 + self.film_strength * torch.tanh(gamma)
        beta = self.film_strength * torch.tanh(beta)

        film_embedding = gamma * cls_embedding + beta

        logits = self.classifier(film_embedding)

        return {
            "logits": logits,
            "log_probs": torch.log_softmax(logits, dim=-1),
            "probs": torch.softmax(logits, dim=-1),
            "gamma": gamma,
            "beta": beta,
        }


model = ContextStrongFiLMRoBERTa(
    model_name=MODEL_NAME,
    num_subgroups=len(subgroup_to_id),
    subgroup_dim=SUBGROUP_DIM,
    film_hidden_dim=FILM_HIDDEN_DIM,
    film_strength=FILM_STRENGTH,
    num_labels=NUM_LABELS,
    dropout=DROPOUT,
).to(DEVICE)

criterion = nn.KLDivLoss(reduction="batchmean")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

num_update_steps_per_epoch = int(np.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS))
num_training_steps = num_update_steps_per_epoch * EPOCHS
num_warmup_steps = int(WARMUP_RATIO * num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

print("Train batches per epoch:", len(train_loader))
print("Training steps:", num_training_steps)
print("Warmup steps:", num_warmup_steps)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Train batches per epoch: 696
Training steps: 4872
Warmup steps: 487


## Metrics

In [17]:
EPS = 1e-12

def kl_divergence(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    y_true = np.clip(y_true, EPS, 1.0)
    y_pred = np.clip(y_pred, EPS, 1.0)
    return np.sum(y_true * np.log(y_true / y_pred), axis=1)

def js_divergence(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    return np.array([
        jensenshannon(true_dist, pred_dist, base=2) ** 2
        for true_dist, pred_dist in zip(y_true, y_pred)
    ])

def cross_entropy(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    y_pred = np.clip(y_pred, EPS, 1.0)
    return -np.sum(y_true * np.log(y_pred), axis=1)

def expected_scores(distributions: np.ndarray) -> np.ndarray:
    labels = np.arange(distributions.shape[1])
    return distributions @ labels

def entropy_values(distributions: np.ndarray) -> np.ndarray:
    return np.array([entropy(dist, base=2) for dist in distributions])

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    true_labels = np.argmax(y_true, axis=1)
    pred_labels = np.argmax(y_pred, axis=1)

    true_expected = expected_scores(y_true)
    pred_expected = expected_scores(y_pred)

    true_entropy = entropy_values(y_true)
    pred_entropy = entropy_values(y_pred)

    metrics = {
        "kl_mean": float(kl_divergence(y_true, y_pred).mean()),
        "js_mean": float(js_divergence(y_true, y_pred).mean()),
        "cross_entropy_mean": float(cross_entropy(y_true, y_pred).mean()),
        "accuracy": float(accuracy_score(true_labels, pred_labels)),
        "macro_f1": float(f1_score(true_labels, pred_labels, average="macro", zero_division=0)),
        "expected_label_mae": float(mean_absolute_error(true_expected, pred_expected)),
    }

    if len(np.unique(true_entropy)) > 1 and len(np.unique(pred_entropy)) > 1:
        metrics["entropy_pearson"] = float(pearsonr(true_entropy, pred_entropy).statistic)
        metrics["entropy_spearman"] = float(spearmanr(true_entropy, pred_entropy).statistic)
    else:
        metrics["entropy_pearson"] = np.nan
        metrics["entropy_spearman"] = np.nan

    return metrics


## Training

In [18]:
def run_epoch(model, dataloader, optimizer=None, scheduler=None):
    is_training = optimizer is not None
    model.train() if is_training else model.eval()

    total_loss = 0.0
    all_targets = []
    all_predictions = []

    if is_training:
        optimizer.zero_grad()

    for step, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        subgroup_id = batch["subgroup_id"].to(DEVICE)
        targets = batch["target_distribution"].to(DEVICE)

        with torch.set_grad_enabled(is_training):
            outputs = model(input_ids, attention_mask, subgroup_id)
            raw_loss = criterion(outputs["log_probs"], targets)

            if is_training:
                loss = raw_loss / GRADIENT_ACCUMULATION_STEPS
                loss.backward()

                should_step = (
                    ((step + 1) % GRADIENT_ACCUMULATION_STEPS == 0)
                    or ((step + 1) == len(dataloader))
                )

                if should_step:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                    optimizer.step()
                    optimizer.zero_grad()
                    if scheduler is not None:
                        scheduler.step()

        total_loss += raw_loss.item() * input_ids.size(0)
        all_targets.append(targets.detach().cpu().numpy())
        all_predictions.append(outputs["probs"].detach().cpu().numpy())

    avg_loss = total_loss / len(dataloader.dataset)
    y_true = np.vstack(all_targets)
    y_pred = np.vstack(all_predictions)

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = float(avg_loss)
    return metrics, y_true, y_pred


best_val_kl = float("inf")
best_model_path = OUTPUT_DIR / "context_strong_film_best_model.pt"
history = []

for epoch in range(1, EPOCHS + 1):
    train_metrics, _, _ = run_epoch(model, train_loader, optimizer=optimizer, scheduler=scheduler)
    val_metrics, _, _ = run_epoch(model, val_loader)

    row = {
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }
    history.append(row)

    print("=" * 80)
    print(f"Epoch {epoch}/{EPOCHS}")
    print(f"Train KL: {train_metrics['kl_mean']:.4f} | Val KL: {val_metrics['kl_mean']:.4f}")
    print(f"Train JS: {train_metrics['js_mean']:.4f} | Val JS: {val_metrics['js_mean']:.4f}")
    print(f"Val Acc: {val_metrics['accuracy']:.4f} | Val Macro F1: {val_metrics['macro_f1']:.4f}")

    if val_metrics["kl_mean"] < best_val_kl:
        best_val_kl = val_metrics["kl_mean"]
        torch.save(model.state_dict(), best_model_path)
        print(f"Saved new best model to {best_model_path}")

history_df = pd.DataFrame(history)
display(history_df)

history_path = OUTPUT_DIR / "context_strong_film_training_history.csv"
history_df.to_csv(history_path, index=False)
print("Saved:", history_path)


Epoch 1/7
Train KL: 0.7157 | Val KL: 0.6831
Train JS: 0.2832 | Val JS: 0.2398
Val Acc: 0.7116 | Val Macro F1: 0.4574
Saved new best model to context_strong_film_outputs/context_strong_film_best_model.pt
Epoch 2/7
Train KL: 0.6104 | Val KL: 0.6670
Train JS: 0.2311 | Val JS: 0.2287
Val Acc: 0.7157 | Val Macro F1: 0.4802
Saved new best model to context_strong_film_outputs/context_strong_film_best_model.pt
Epoch 3/7
Train KL: 0.5755 | Val KL: 0.6557
Train JS: 0.2138 | Val JS: 0.2327
Val Acc: 0.7190 | Val Macro F1: 0.4751
Saved new best model to context_strong_film_outputs/context_strong_film_best_model.pt
Epoch 4/7
Train KL: 0.5408 | Val KL: 0.6962
Train JS: 0.2024 | Val JS: 0.2288
Val Acc: 0.7247 | Val Macro F1: 0.4826
Epoch 5/7
Train KL: 0.4953 | Val KL: 0.7187
Train JS: 0.1853 | Val JS: 0.2327
Val Acc: 0.7132 | Val Macro F1: 0.4742
Epoch 6/7
Train KL: 0.4543 | Val KL: 0.7163
Train JS: 0.1693 | Val JS: 0.2343
Val Acc: 0.7230 | Val Macro F1: 0.4780
Epoch 7/7
Train KL: 0.4100 | Val KL: 0.7

,epoch,train_kl_mean,train_js_mean,train_cross_entropy_mean,train_accuracy,train_macro_f1,train_expected_label_mae,train_entropy_pearson,train_entropy_spearman,train_loss,val_kl_mean,val_js_mean,val_cross_entropy_mean,val_accuracy,val_macro_f1,val_expected_label_mae,val_entropy_pearson,val_entropy_spearman,val_loss
0,1,0.715671,0.283197,0.796277,0.658760,0.407150,0.647862,0.083270,0.067406,0.715671,0.683095,0.239779,0.756225,0.711601,0.457422,0.519092,0.102024,0.101155,0.683095
1,2,0.610402,0.231130,0.691008,0.728302,0.474525,0.521954,0.144717,0.132672,0.610402,0.667038,0.228725,0.740168,0.715686,0.480155,0.497943,0.118251,0.103531,0.667038
2,3,0.575544,0.213766,0.656150,0.746451,0.488110,0.476510,0.171996,0.164741,0.575544,0.655741,0.232711,0.728871,0.718954,0.475149,0.500594,0.114030,0.107893,0.655741
3,4,0.540811,0.202380,0.621417,0.762444,0.500723,0.446935,0.202426,0.193973,0.540811,0.696153,0.228816,0.769283,0.724673,0.482582,0.494321,0.099628,0.097322,0.696153
4,5,0.495297,0.185296,0.575903,0.782929,0.516694,0.401109,0.245838,0.236759,0.495297,0.718722,0.232689,0.791852,0.713235,0.474185,0.498937,0.105698,0.106253,0.718722
5,6,0.454342,0.169274,0.534948,0.794070,0.527897,0.362335,0.262658,0.254415,0.454342,0.716312,0.234288,0.789442,0.723039,0.478017,0.497663,0.118642,0.117361,0.716312
6,7,0.409955,0.154132,0.490561,0.815274,0.581871,0.327935,0.286869,0.279344,0.409955,0.756738,0.235742,0.829867,0.722222,0.474648,0.492315,0.120651,0.126000,0.756738


Saved: context_strong_film_outputs/context_strong_film_training_history.csv


## Test evaluation and predictions

In [19]:
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))

test_metrics, y_true_test, y_pred_test = run_epoch(model, test_loader)

display(test_metrics)

metrics_path = OUTPUT_DIR / "context_strong_film_test_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(test_metrics, f, indent=2)
print("Saved:", metrics_path)

predictions_df = test_df.copy()
predictions_df["true_distribution"] = list(y_true_test)
predictions_df["pred_distribution"] = list(y_pred_test)
predictions_df["pred_majority_label"] = np.argmax(y_pred_test, axis=1)
predictions_df["pred_expected_label"] = expected_scores(y_pred_test)
predictions_df["pred_entropy"] = entropy_values(y_pred_test)
predictions_df["kl"] = kl_divergence(y_true_test, y_pred_test)
predictions_df["js"] = js_divergence(y_true_test, y_pred_test)
predictions_df["cross_entropy"] = cross_entropy(y_true_test, y_pred_test)

predictions_path = OUTPUT_DIR / "context_strong_film_test_predictions.parquet"
predictions_df.to_parquet(predictions_path, index=False)
predictions_df.to_csv(OUTPUT_DIR / "context_strong_film_test_predictions.csv", index=False)
print("Saved:", predictions_path)


/tmp/ipykernel_58651/352382768.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))


{'kl_mean': 0.7195029258728027,
 'js_mean': 0.2449340286400218,
 'cross_entropy_mean': 0.8029659986495972,
 'accuracy': 0.7024722932651322,
 'macro_f1': 0.4584038047858218,
 'expected_label_mae': 0.5408951015976231,
 'entropy_pearson': 0.11864825934386188,
 'entropy_spearman': 0.11715915519684982,
 'loss': 0.7195029367977802}

Saved: context_strong_film_outputs/context_strong_film_test_metrics.json
Saved: context_strong_film_outputs/context_strong_film_test_predictions.parquet


## Diagnostics

In [20]:
true_labels = np.argmax(y_true_test, axis=1)
pred_labels = np.argmax(y_pred_test, axis=1)

print("Confusion matrix:")
print(confusion_matrix(true_labels, pred_labels, labels=[0, 1, 2]))

print("\nClassification report:")
print(classification_report(true_labels, pred_labels, labels=[0, 1, 2], zero_division=0))

print("\nPredicted label distribution:")
display(pd.Series(pred_labels).value_counts(normalize=True).sort_index())

print("\nTrue label distribution:")
display(pd.Series(true_labels).value_counts(normalize=True).sort_index())

print("\nAverage predicted distribution:")
print(np.vstack(predictions_df["pred_distribution"].to_numpy()).mean(axis=0))

print("\nMean KL by true majority label:")
display(
    predictions_df
    .groupby("target_majority_label")
    .agg(
        n=("comment_id", "count"),
        mean_kl=("kl", "mean"),
        mean_js=("js", "mean"),
        mean_target_entropy=("target_distribution", lambda x: np.mean([entropy(parse_distribution(v), base=2) for v in x])),
        mean_pred_entropy=("pred_entropy", "mean"),
    )
)

print("Average predicted distribution by true majority label:")
for label in [0, 1, 2]:
    subset = predictions_df[predictions_df["target_majority_label"] == label]
    avg_pred = np.vstack(subset["pred_distribution"].to_numpy()).mean(axis=0)
    print(label, avg_pred)


Confusion matrix:
[[622   0 163]
 [ 55   0  31]
 [100   0 202]]

Classification report:
              precision    recall  f1-score   support

           0       0.80      0.79      0.80       785
           1       0.00      0.00      0.00        86
           2       0.51      0.67      0.58       302

    accuracy                           0.70      1173
   macro avg       0.44      0.49      0.46      1173
weighted avg       0.67      0.70      0.68      1173


Predicted label distribution:


0    0.662404
2    0.337596
Name: proportion, dtype: float64


True label distribution:


0    0.669224
1    0.073316
2    0.257460
Name: proportion, dtype: float64


Average predicted distribution:
[0.6602004  0.07085895 0.26894066]

Mean KL by true majority label:


,n,mean_kl,mean_js,mean_target_entropy,mean_pred_entropy
target_majority_label,,,,,
0,785,0.380558,0.146538,0.142549,0.657905
1,86,2.465198,0.722004,0.197674,0.913810
2,302,1.103418,0.364844,0.040868,1.072843


Average predicted distribution by true majority label:
0 [0.76048726 0.06007381 0.17943889]
1 [0.6068132  0.08545554 0.30773115]
2 [0.41472405 0.09473664 0.49053946]


In [21]:
print("Performance by subgroup:")

subgroup_rows = []

for subgroup, group in predictions_df.groupby("subgroup"):
    y_true = np.vstack(group["true_distribution"].to_numpy())
    y_pred = np.vstack(group["pred_distribution"].to_numpy())

    row = {
        "subgroup": subgroup,
        "n": len(group),
        **compute_metrics(y_true, y_pred),
    }
    subgroup_rows.append(row)

subgroup_metrics_df = pd.DataFrame(subgroup_rows).sort_values("kl_mean")
display(subgroup_metrics_df)

subgroup_metrics_path = OUTPUT_DIR / "context_strong_film_subgroup_metrics.csv"
subgroup_metrics_df.to_csv(subgroup_metrics_path, index=False)
print("Saved:", subgroup_metrics_path)


Performance by subgroup:


,subgroup,n,kl_mean,js_mean,cross_entropy_mean,accuracy,macro_f1,expected_label_mae,entropy_pearson,entropy_spearman
1,non_binary,16,0.525606,0.199437,0.525606,0.812500,0.539855,0.442971,NaN,NaN
0,men,576,0.715348,0.239319,0.807942,0.697917,0.441779,0.535892,0.166933,0.146877
2,women,581,0.728962,0.251753,0.805671,0.703959,0.469939,0.548552,0.065419,0.084444


Saved: context_strong_film_outputs/context_strong_film_subgroup_metrics.csv


## Counterfactual subgroup sensitivity

In [22]:
@torch.no_grad()
def predict_distribution(context_text: str, subgroup: str) -> np.ndarray:
    model.eval()

    encoding = tokenizer(
        context_text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    input_ids = encoding["input_ids"].to(DEVICE)
    attention_mask = encoding["attention_mask"].to(DEVICE)

    subgroup_id = torch.tensor([subgroup_to_id[subgroup]], dtype=torch.long).to(DEVICE)

    outputs = model(input_ids, attention_mask, subgroup_id)
    return outputs["probs"].detach().cpu().numpy()[0]


subgroups = list(subgroup_to_id.keys())

context_lookup = {
    (row["comment_id"], row["subgroup"]): row["context_input_text"]
    for _, row in context_df.iterrows()
}

unique_test_comments = test_df[["comment_id", "text"]].drop_duplicates("comment_id").reset_index(drop=True)

counterfactual_rows = []

for _, row in unique_test_comments.iterrows():
    comment_id = row["comment_id"]

    available_subgroups = [
        subgroup for subgroup in subgroups
        if (comment_id, subgroup) in context_lookup
    ]

    if len(available_subgroups) < 2:
        continue

    preds_by_group = {}
    for subgroup in available_subgroups:
        context_text = context_lookup[(comment_id, subgroup)]
        preds_by_group[subgroup] = predict_distribution(context_text, subgroup)

    pairwise_js = []
    for group_a, group_b in itertools.combinations(available_subgroups, 2):
        js = jensenshannon(preds_by_group[group_a], preds_by_group[group_b], base=2) ** 2
        pairwise_js.append(js)

    counterfactual_rows.append({
        "comment_id": comment_id,
        "text": row["text"],
        "n_available_subgroups": len(available_subgroups),
        "mean_pairwise_js": float(np.mean(pairwise_js)),
        "max_pairwise_js": float(np.max(pairwise_js)),
    })

cf_df = pd.DataFrame(counterfactual_rows)

print("Counterfactual rows:", cf_df.shape)
print("Mean counterfactual JS:", cf_df["mean_pairwise_js"].mean())
print("Median counterfactual JS:", cf_df["mean_pairwise_js"].median())
print("Mean max-pairwise JS:", cf_df["max_pairwise_js"].mean())
print("Median max-pairwise JS:", cf_df["max_pairwise_js"].median())

display(cf_df.sort_values("max_pairwise_js", ascending=False).head(30))

cf_path = OUTPUT_DIR / "context_strong_film_counterfactual_js.csv"
cf_df.to_csv(cf_path, index=False)
print("Saved:", cf_path)


Counterfactual rows: (582, 5)
Mean counterfactual JS: 0.0058378357565340416
Median counterfactual JS: 0.004134980218308968
Mean max-pairwise JS: 0.0058902151811064075
Median max-pairwise JS: 0.004134980218308968


,comment_id,text,n_available_subgroups,mean_pairwise_js,max_pairwise_js
559,47822,Survey says women are emotional lunatics.,2,0.092511,0.092511
455,35197,Where the single friendly lesbians at ? Not the ones wit gfs in whole relationships an pregnant tf,2,0.072344,0.072344
365,27814,"Can't wait until the suburban white girls run the ""bitch I'll kill you"" format into the ground",2,0.057705,0.057705
21,1572,I guess a Pussy Hat isn't the stupidest thing I've ever seen on a woman's head.,2,0.057445,0.057445
547,45258,"These 4 women are hate mongers and racists, they are actually bullying the dems. Members ,they are scared of them, these women need to be spanked! Maga",2,0.053081,0.053081
558,47565,Crazy women with very tiny little brains trying to destroy yet another western country!,2,0.048541,0.048541
62,3696,What a hot little fuck puppet she is! Would fuck her in the ass all night long!,2,0.037626,0.037626
550,45821,"OMG. What a babbling, clearly insane woman. Those psychotic hand gestures. Those delusional, lying eyes. That robotic voice. This woman is not well. She's needs to be removed from her position at once and given a tho...",2,0.034634,0.034634
536,44042,She's so retarded,2,0.030725,0.030725
216,14894,Yeah dont i know it back in my day a woman didnt even have the right to say no and if she did youd give her a good wallop and these millenials woth tgeir phones and degrees buch of fucking nancies dont i know it jim ...,2,0.030077,0.030077


Saved: context_strong_film_outputs/context_strong_film_counterfactual_js.csv


## EL vs EC counterfactual analysis

In [26]:
def pairwise_counterfactual_js(group_a: str, group_b: str) -> pd.DataFrame:
    rows = []
    unique_comments = test_df[["comment_id", "text"]].drop_duplicates("comment_id").reset_index(drop=True)

    for _, row in unique_comments.iterrows():
        comment_id = row["comment_id"]

        if (comment_id, group_a) not in context_lookup:
            continue
        if (comment_id, group_b) not in context_lookup:
            continue

        context_a = context_lookup[(comment_id, group_a)]
        context_b = context_lookup[(comment_id, group_b)]

        pred_a = predict_distribution(context_a, group_a)
        pred_b = predict_distribution(context_b, group_b)

        js = jensenshannon(pred_a, pred_b, base=2) ** 2

        rows.append({
            "comment_id": comment_id,
            "text": row["text"],
            "group_a": group_a,
            "group_b": group_b,
            "js": float(js),
            f"pred_{group_a}": pred_a,
            f"pred_{group_b}": pred_b,
            f"context_{group_a}": context_a,
            f"context_{group_b}": context_b,
        })

    return pd.DataFrame(rows)


if "men" in subgroup_to_id and "women" in subgroup_to_id:
    extreme_df = pairwise_counterfactual_js("men", "women")

    print("M vs W rows:", extreme_df.shape)
    print("Men vs Women mean JS:", extreme_df["js"].mean())
    print("Men vs Women median JS:", extreme_df["js"].median())

    display(extreme_df.sort_values("js", ascending=False).head(30))

    extreme_path = OUTPUT_DIR / "context_strong_film_M_vs_W_counterfactual.csv"
    extreme_df.to_csv(extreme_path, index=False)
    print("Saved:", extreme_path)


M vs W rows: (575, 9)
Men vs Women mean JS: 0.0058057145286201884
Men vs Women median JS: 0.004155393752797706


,comment_id,text,group_a,group_b,js,pred_men,pred_women,context_men,context_women
553,47822,Survey says women are emotional lunatics.,men,women,0.092511,"[0.73551565, 0.089112245, 0.17537221]","[0.3897866, 0.14811984, 0.46209362]","### COMMENT TO CLASSIFY\nSurvey says women are emotional lunatics.\n\n### ANNOTATOR GENDER\nmen\n\n### RETRIEVED BACKGROUND\nGender role:\nGender roles are social norms that vary across cultures, influencing a wide r...",### COMMENT TO CLASSIFY\nSurvey says women are emotional lunatics.\n\n### ANNOTATOR GENDER\nwomen\n\n### RETRIEVED BACKGROUND\nMisogyny:\nMisogyny is a form of sexism that perpetuates women's subordinate status in pa...
450,35197,Where the single friendly lesbians at ? Not the ones wit gfs in whole relationships an pregnant tf,men,women,0.072344,"[0.8696919, 0.05416884, 0.076139316]","[0.5952181, 0.14725573, 0.2575262]",### COMMENT TO CLASSIFY\nWhere the single friendly lesbians at ? Not the ones wit gfs in whole relationships an pregnant tf\n\n### ANNOTATOR GENDER\nmen\n\n### RETRIEVED BACKGROUND\nGender role:\nGender roles are soc...,### COMMENT TO CLASSIFY\nWhere the single friendly lesbians at ? Not the ones wit gfs in whole relationships an pregnant tf\n\n### ANNOTATOR GENDER\nwomen\n\n### RETRIEVED BACKGROUND\nMisogyny:\nMisogyny is a form of...
361,27814,"Can't wait until the suburban white girls run the ""bitch I'll kill you"" format into the ground",men,women,0.057705,"[0.64379096, 0.105386846, 0.25082213]","[0.37055305, 0.13297553, 0.49647143]","### COMMENT TO CLASSIFY\nCan't wait until the suburban white girls run the ""bitch I'll kill you"" format into the ground\n\n### ANNOTATOR GENDER\nmen\n\n### RETRIEVED BACKGROUND\nMale privilege:\nIn patriarchal societ...","### COMMENT TO CLASSIFY\nCan't wait until the suburban white girls run the ""bitch I'll kill you"" format into the ground\n\n### ANNOTATOR GENDER\nwomen\n\n### RETRIEVED BACKGROUND\nViolence against women:\nViolence ag..."
21,1572,I guess a Pussy Hat isn't the stupidest thing I've ever seen on a woman's head.,men,women,0.057445,"[0.8339807, 0.06274827, 0.10327108]","[0.5804246, 0.15373614, 0.26583925]","### COMMENT TO CLASSIFY\nI guess a Pussy Hat isn't the stupidest thing I've ever seen on a woman's head.\n\n### ANNOTATOR GENDER\nmen\n\n### RETRIEVED BACKGROUND\nMasculinity:\nIn Western cultures, traditional mascul...",### COMMENT TO CLASSIFY\nI guess a Pussy Hat isn't the stupidest thing I've ever seen on a woman's head.\n\n### ANNOTATOR GENDER\nwomen\n\n### RETRIEVED BACKGROUND\nMisogyny:\nMisogyny is a form of sexism that perpet...
552,47565,Crazy women with very tiny little brains trying to destroy yet another western country!,men,women,0.048541,"[0.8974914, 0.04007535, 0.062433247]","[0.6918468, 0.12636788, 0.18178535]","### COMMENT TO CLASSIFY\nCrazy women with very tiny little brains trying to destroy yet another western country!\n\n### ANNOTATOR GENDER\nmen\n\n### RETRIEVED BACKGROUND\nMale privilege:\nIn patriarchal societies, ma...",### COMMENT TO CLASSIFY\nCrazy women with very tiny little brains trying to destroy yet another western country!\n\n### ANNOTATOR GENDER\nwomen\n\n### RETRIEVED BACKGROUND\nViolence against women:\nViolence against w...
61,3696,What a hot little fuck puppet she is! Would fuck her in the ass all night long!,men,women,0.037626,"[0.5681725, 0.13301535, 0.29881218]","[0.34940788, 0.15262413, 0.49796802]","### COMMENT TO CLASSIFY\nWhat a hot little fuck puppet she is! Would fuck her in the ass all night long!\n\n### ANNOTATOR GENDER\nmen\n\n### RETRIEVED BACKGROUND\nMasculinity:\nIn Western cultures, traditional mascul...",### COMMENT TO CLASSIFY\nWhat a hot little fuck puppet she is! Would fuck her in the ass all night long!\n\n### ANNOTATOR GENDER\nwomen\n\n### RETRIEVED BACKGROUND\nViolence against women:\nViolence against women is ...
544,45821,"OMG. What a babbling, clearly insane woman. Those psychotic hand gestures. Those delusional, lying eyes. That robotic voice. This 

Saved: context_strong_film_outputs/context_strong_film_M_vs_W_counterfactual.csv


## True EL/EC disagreement vs model disagreement

In [29]:
def valid_dist(x):
    if x is None:
        return False
    try:
        arr = np.array(parse_distribution(x), dtype=float)
    except Exception:
        return False
    if arr.shape[0] != NUM_LABELS:
        return False
    if np.isnan(arr).any():
        return False
    if arr.sum() <= 0:
        return False
    return True


def true_pairwise_disagreement_from_context_df(full_df: pd.DataFrame, group_a: str, group_b: str) -> pd.DataFrame:
    rows = []

    for comment_id, group in full_df.groupby("comment_id"):
        if group_a not in set(group["subgroup"]):
            continue
        if group_b not in set(group["subgroup"]):
            continue

        row_a = group[group["subgroup"] == group_a].iloc[0]
        row_b = group[group["subgroup"] == group_b].iloc[0]

        dist_a = parse_distribution(row_a["target_distribution"])
        dist_b = parse_distribution(row_b["target_distribution"])

        if not valid_dist(dist_a) or not valid_dist(dist_b):
            continue

        dist_a = np.array(dist_a, dtype=float)
        dist_b = np.array(dist_b, dtype=float)
        true_js = jensenshannon(dist_a, dist_b, base=2) ** 2

        rows.append({
            "comment_id": comment_id,
            "text": row_a["text"],
            "true_js": float(true_js),
            f"{group_a}_true_dist": dist_a,
            f"{group_b}_true_dist": dist_b,
            f"{group_a}_true_label": int(np.argmax(dist_a)),
            f"{group_b}_true_label": int(np.argmax(dist_b)),
        })

    return pd.DataFrame(rows)


if "men" in subgroup_to_id and "women" in subgroup_to_id:
    true_df = true_pairwise_disagreement_from_context_df(context_df, "men", "women")

    comparison_df = true_df.merge(
        extreme_df[[
            "comment_id",
            "js",
            "pred_men",
            "pred_women",
        ]],
        on="comment_id",
        how="inner",
    ).rename(columns={"js": "model_js"})

    nonzero = comparison_df[comparison_df["true_js"] > 0].copy()
    nonzero["capture_ratio"] = nonzero["model_js"] / nonzero["true_js"]

    print("N true men/women overlap:", len(true_df))
    print("N comparison:", len(comparison_df))
    print("Mean true JS:", comparison_df["true_js"].mean())
    print("Mean model JS:", comparison_df["model_js"].mean())

    if len(nonzero) > 0:
        print("Mean capture ratio, true_js > 0:", nonzero["capture_ratio"].mean())
        print("Median capture ratio, true_js > 0:", nonzero["capture_ratio"].median())

    comparison_df["label_pair"] = (
        comparison_df["men_true_label"].astype(str)
        + "-"
        + comparison_df["women_true_label"].astype(str)
    )

    display(
        comparison_df
        .groupby("label_pair")
        .agg(
            n=("comment_id", "count"),
            mean_true_js=("true_js", "mean"),
            mean_model_js=("model_js", "mean"),
        )
        .sort_values("mean_true_js", ascending=False)
    )

    display(comparison_df.sort_values("true_js", ascending=False).head(30))

    comparison_path = OUTPUT_DIR / "context_strong_film_true_vs_model_men_women.csv"
    comparison_df.to_csv(comparison_path, index=False)
    print("Saved:", comparison_path)


N true men/women overlap: 3869
N comparison: 575
Mean true JS: 0.34650682259322935
Mean model JS: 0.0058057145286201884
Mean capture ratio, true_js > 0: 0.025095375580249456
Median capture ratio, true_js > 0: 0.007504306775217695


,n,mean_true_js,mean_model_js
label_pair,,,
1-0,21,1.000000,0.006457
2-1,7,1.000000,0.009765
0-1,32,0.962852,0.007702
2-0,50,0.865213,0.007106
0-2,84,0.828284,0.007094
1-2,12,0.827820,0.006192
0-0,291,0.055885,0.004566
1-1,6,0.051880,0.008456
2-2,72,0.015082,0.006710


,comment_id,text,true_js,men_true_dist,women_true_dist,men_true_label,women_true_label,model_js,pred_men,pred_women,label_pair
480,38410,to all those ugly hoes complaining about the recent unedited photoshoot of nct. Fuck off you should look at yourself. They are also fucking humans with feelings and skin problems. for example I also don't have clear ...,1.0,"[0.0, 1.0, 0.0]","[1.0, 0.0, 0.0]",1,0,0.003565,"[0.46540877, 0.13588819, 0.398703]","[0.39584503, 0.15482448, 0.44933048]",1-0
183,12324,Now she most certainly is on tape saying she wanted to grab women by the pussy right? No?,1.0,"[1.0, 0.0, 0.0]","[0.0, 1.0, 0.0]",0,1,0.001342,"[0.9734654, 0.013947825, 0.012586883]","[0.958079, 0.023756674, 0.018164419]",0-1
478,38126,"Y'all always slip up and say how y'all see women period because this includes the majority of women in yall lives that's not related to you. Especially since men demand pussy just from taking you out, selling you wee...",1.0,"[1.0, 0.0, 0.0]","[0.0, 1.0, 0.0]",0,1,0.003559,"[0.34819725, 0.11896755, 0.5328352]","[0.2840454, 0.122088805, 0.59386575]",0-1
192,13332,"I'd fuck you like a bitch, pulling your hair and spanking your ass as I stretch your tight little hole",1.0,"[1.0, 0.0, 0.0]","[0.0, 1.0, 0.0]",0,1,0.000964,"[0.9359834, 0.030499956, 0.033516686]","[0.92238194, 0.044332776, 0.03328525]",0-1
476,37948,"Ya the prime example of an unloyal Girlfriend you, and a fuckin pussy",1.0,"[0.6666666865348816, 0.0, 0.3333333432674408]","[0.0, 1.0, 0.0]",0,1,0.004928,"[0.49278724, 0.15974253, 0.34747025]","[0.49952117, 0.2136542, 0.28682464]",0-1
194,13371,women enjoy retarded expressions of dumb niceness,1.0,"[1.0, 0.0, 0.0]","[0.0, 0.0, 1.0]",0,2,0.007573,"[0.9279258, 0.03316588, 0.03890834]","[0.8665048, 0.06638933, 0.06710589]",0-2
196,13431,I'd spin you around and slam you on this dick like a real fuck toy.,1.0,"[0.0, 0.0, 1.0]","[1.0, 0.0, 0.0]",2,0,0.003849,"[0.9295546, 0.03627693, 0.034168527]","[0.8908592, 0.066141, 0.04299987]",2-0
474,37899,"U hoes go out 2 Eat with anybody, you gone get kidnapped 1 day u hungry bitch",1.0,"[0.0, 0.0, 1.0]","[1.0, 0.0, 0.0]",2,0,0.008499,"[0.28341144, 0.104607224, 0.61198133]","[0.19745688, 0.09215653, 0.7103866]",2-0
201,13955,Depression Stress wait no women have those too yea this guys a dickbag,1.0,"[0.0, 0.0, 1.0]","[1.0, 0.0, 0.0]",2,0,0.001442,"[0.52579325, 0.13998151, 0.33422518]","[0.50801414, 0.17237481, 0.31961104]",2-0
203,14011,Plant her face in my wife's pussy while I fuck her from behind,1.0,"[0.0, 1.0, 0.0]","[1.0, 0.0, 0.0]",1,0,0.001227,"[0.9671783, 0.017235791, 0.015585947]","[0.9516051, 0.02824892, 0.020146035]",1-0


Saved: context_strong_film_outputs/context_strong_film_true_vs_model_men_women.csv


## Inspect highest EL/EC context effect

In [31]:
if "extreme_df" in globals() and len(extreme_df) > 0:
    top_row = extreme_df.sort_values("js", ascending=False).iloc[0]

    print("Comment:")
    print(top_row["text"])

    print("\nPrediction men:")
    print(top_row["pred_men"])

    print("\nPrediction women:")
    print(top_row["pred_women"])

    print("\nContext men:")
    print(top_row["context_men"])

    print("\nContext women:")
    print(top_row["context_women"])

Comment:
Survey says women are emotional lunatics.

Prediction men:
[0.73551565 0.08911224 0.17537221]

Prediction women:
[0.3897866  0.14811984 0.46209362]

Context men:
### COMMENT TO CLASSIFY
Survey says women are emotional lunatics.

### ANNOTATOR GENDER
men

### RETRIEVED BACKGROUND
Gender role:
Gender roles are social norms that vary across cultures, influencing a wide range of human behavior, including profession, relationships, and personal expression. Traditionally, women have been confined to the "private" sphere while men occupy the "public" sphere. Sociologists distinguish between gender role and sex identity, with the former being shaped by societal expectations and the latter being an individual's internal sense of their own gender. Theories such as social constructionism argue that differences in behavior are due to cultural and social factors rather than biological or essentialist ones.

Context women:
### COMMENT TO CLASSIFY
Survey says women are emotional lunatics.

#

## Interpretation Guide

Compare this model against:

1. Embedding baseline, no context
2. Strong FiLM, no context
3. Context + Embedding
4. Context + Strong FiLM

Most important metrics:

```text
KL / JS
Macro F1
Class-2 recall
Mean counterfactual JS
EL↔EC model JS
True-vs-model capture ratio
```
